# 🧹 Cafe Sales — Data Cleaning Pipeline

## Project Overview

This notebook cleans the **Cafe Sales Dirty Dataset** sourced from Kaggle via the Kaggle API.

The dataset contains **10,000 rows** of intentionally corrupted cafe sales transactions, including:
- Missing values (`NaN`, `None`)
- Invalid string entries (`"ERROR"`, `"UNKNOWN"`)
- Incorrect or missing numeric values across related columns
- Missing and unparsed dates

### Cleaning Strategy
Each column is treated based on the nature of its data and the type of corruption present. Where possible, missing values are **calculated mathematically** from related columns. Where ambiguity exists, **random sampling** is used to preserve the natural distribution. Only values that cannot be recovered by any method are **dropped**.

### Pipeline Steps
1. Ingest dataset via Kaggle API
2. Explore raw dirty data
3. Replace invalid string entries with `NaN`
4. Cast columns to correct data types
5. Impute missing numeric values using mathematical relationships
6. Impute missing `Item` values using price mapping and random sampling
7. Impute missing categorical values using random sampling
8. Interpolate missing dates
9. Final pass — drop irrecoverable rows
10. Report data loss summary

## Step 1 — Download Dataset via Kaggle API

The dataset is downloaded programmatically using the Kaggle API rather than manually. This keeps the pipeline reproducible — anyone with a Kaggle account can run this notebook end to end without downloading files manually.

Credentials are managed via Colab Secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`) so no sensitive information is hardcoded into the notebook.

In [ ]:
!kaggle datasets download -d ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

Dataset URL: https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training
License(s): CC-BY-SA-4.0
100% 111k/111k [00:00<00:00, 70.3MB/s]



## Step 2 — Unzip Dataset

The downloaded file is a `.zip` archive. We extract it into `/content/data/` to access the raw CSV file.

In [ ]:
!unzip cafe-sales-dirty-data-for-cleaning-training.zip -d /content/data

Archive:  cafe-sales-dirty-data-for-cleaning-training.zip
  inflating: /content/data/dirty_cafe_sales.csv  


## Step 3 — Import Libraries

| Library | Purpose |
|---|---|
| `pandas` | Data manipulation and cleaning |
| `numpy` | Numerical operations and `NaN` handling |

| `warnings` | Suppress non-critical pandas warnings during cleaning |

In [ ]:
# Standard Imports
import pandas as pd
import numpy as np

# Utilities
import warnings
warnings.filterwarnings('ignore')

## Step 4 — Load Dataset

The CSV is loaded into a pandas DataFrame. Crucially, we immediately create a **working copy** (`df`) and keep the original (`original_data`) untouched.

This is a critical practice — it allows us to:
- Compare the cleaned data against the original at any point
- Calculate data loss at the end
- Restart cleaning without re-downloading the file

In [ ]:
original_data=pd.read_csv("/content/data/dirty_cafe_sales.csv")
df=original_data.copy()
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


## Step 5 — Initial Exploration

Before any cleaning, we profile the raw dataset to understand its structure and identify what needs to be fixed.

The `summ_and_report()` function reports:
- **Shape** — number of rows and columns
- **Duplicates** — any repeated transactions
- **Data types** — whether columns are the correct type
- **Descriptive statistics** — min, max, mean, and distribution of numeric columns

This step defines the problem before we attempt any solution.

In [ ]:
def summ_and_report(df):

    # Shape of Data
    print('\nNumber of Rows in Dataset       :\t', df.shape[0])
    print('\nNumber of Columns in Dataset    :\t', df.shape[1])

    # Number of Duplicates in Data
    print('\nNumber of Duplicated in Dataset :\t',df.duplicated().sum())

    # Information of Dataset
    print('\nInformation Of Dataset :')
    display(df.info())

    # Description of Data
    print('\nDescription of Numerical Data :')
    display(df.describe())

summ_and_report(df)


Number of Rows in Dataset       :	 10000

Number of Columns in Dataset    :	 8

Number of Duplicated in Dataset :	 0

Information Of Dataset :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


None


Description of Numerical Data :


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


# ****Column Descriptions :****

| 🏷️ **Column Name**     | 📝 **Description**                                                                          | 🔍 **Example Values**  |
| ----------------------- | ------------------------------------------------------------------------------------------- | ---------------------- |
| 🆔 **Transaction ID**   | Unique identifier for each transaction. Always present and unique.                          | `TXN_1234567`          |
| 🛒 **Item**             | Name of the purchased item. May contain missing or invalid values such as `"ERROR"`.        | `Coffee`, `Sandwich`   |
| 🔢 **Quantity**         | Number of items purchased. May include missing or invalid values.                           | `1`, `3`, `UNKNOWN`    |
| 💲 **Price Per Unit**   | Cost of a single unit of the item. May contain missing or invalid values.                   | `2.00`, `4.00`         |
| 💰 **Total Spent**      | Total transaction amount calculated as **Quantity × Price Per Unit**.                       | `8.00`, `12.00`        |
| 💳 **Payment Method**   | Method used for payment. May include missing or invalid entries like `None` or `"UNKNOWN"`. | `Cash`, `Credit Card`  |
| 📍 **Location**         | Where the transaction occurred. May contain missing or invalid values.                      | `In-store`, `Takeaway` |
| 📅 **Transaction Date** | Date when the transaction happened. May include missing or incorrect values.                | `2023-01-01`           |


## Step 6 — Inspect Unique Values Per Column

We loop through every column and display its unique values. This reveals:
- Invalid string entries such as `"ERROR"` and `"UNKNOWN"`
- Unexpected categories or casing inconsistencies
- Which columns contain mixed types (e.g. numeric columns holding string errors)

This inspection directly informs which values need to be replaced with `NaN` in the next step.

In [ ]:
for i in df.columns:
    print('\n', '=' * 80, '\n')
    display(df[i].unique())

array(['TXN_1961373', 'TXN_4977031', 'TXN_4271903', ..., 'TXN_5255387',
       'TXN_7695629', 'TXN_6170729'], dtype=object)

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

array(['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan], dtype=object)

array(['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan, 'ERROR', 'UNKNOWN'],
      dtype=object)

array(['4.0', '12.0', 'ERROR', '10.0', '20.0', '9.0', '16.0', '15.0',
       '25.0', '8.0', '5.0', '3.0', '6.0', nan, 'UNKNOWN', '2.0', '1.0',
       '7.5', '4.5', '1.5'], dtype=object)

array(['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan],
      dtype=object)

array(['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR'], dtype=object)

array(['2023-09-08', '2023-05-16', '2023-07-19', '2023-04-27',
       '2023-06-11', '2023-03-31', '2023-10-06', '2023-10-28',
       '2023-07-28', '2023-12-31', '2023-11-07', 'ERROR', '2023-05-03',
       '2023-06-01', '2023-03-21', '2023-11-15', '2023-06-10',
       '2023-02-24', '2023-03-25', '2023-01-15', '2023-04-04',
       '2023-03-30', '2023-12-01', '2023-09-18', '2023-06-03',
       '2023-12-13', '2023-04-20', '2023-04-10', '2023-03-11',
       '2023-06-02', '2023-11-06', '2023-08-15', '2023-10-09',
       '2023-05-28', '2023-07-17', '2023-04-29', '2023-06-08',
       '2023-06-29', '2023-04-17', '2023-12-22', '2023-01-10',
       '2023-10-02', '2023-02-23', '2023-03-22', '2023-11-03',
       '2023-03-02', '2023-06-26', '2023-05-02', '2023-09-05',
       '2023-01-08', '2023-03-15', '2023-11-25', '2023-12-05',
       '2023-03-19', '2023-06-27', '2023-04-19', '2023-10-07',
       '2023-09-30', '2023-05-27', '2023-11-18', '2023-10-20',
       '2023-10-03', '2023-10-27', '2023-04-06

## Step 7 — Replace Invalid Entries and Cast Data Types

### 7.1 — Replace `"ERROR"` and `"UNKNOWN"` with `NaN`
The strings `"ERROR"` and `"UNKNOWN"` are not real values — they are placeholders for missing data. We replace them with `np.nan` so that pandas recognises them as missing and all null-handling functions work correctly.

> Using `np.nan` is important here. If we replaced with an empty string `""` instead, functions like `isnull()`, `dropna()`, and `fillna()` would not detect them as missing.

### 7.2 — Cast Numeric Columns to `float64`
After replacing invalid strings, the `Quantity`, `Price Per Unit`, and `Total Spent` columns still hold mixed types. We cast them to `float64` so mathematical operations can be performed on them.

### 7.3 — Parse `Transaction Date` to datetime
The date column is stored as plain text. We convert it to a proper `datetime` type so pandas can perform date arithmetic, interpolation, and feature extraction on it.

In [ ]:
df.replace(["ERROR","UNKNOWN"],np.nan,inplace=True)
df[['Quantity','Price Per Unit','Total Spent']]=df[['Quantity','Price Per Unit','Total Spent']].astype(np.float64)
df['Transaction Date']=pd.to_datetime(df['Transaction Date'])

## ****Calculate null values in `Quantity`, `Price Per Unit`, `Total Spent`****

- We can use formula **`Total Spent`** = **`Quantity`** * **`Price Per Unit`** to calculate missing values all three columns.

### Numeric Imputation — Mathematical Relationship

The three columns `Quantity`, `Price Per Unit`, and `Total Spent` share a fixed mathematical relationship:

```
Total Spent = Quantity × Price Per Unit
```

This means if any two of the three are known, the third can be calculated exactly — no guessing required.

**Order matters:** `Quantity` and `Price Per Unit` are filled first. By the time we calculate `Total Spent`, some previously missing values in those columns have already been recovered, allowing more `Total Spent` values to be calculated in turn.

In [ ]:
def null_calculator(df):
    df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
    df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])
    df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

null_calculator(df)

## ****Calculate Null in `Item` Column****

- In this column we can calculate missing data using Price Per Item.
- But there is catch that `Sandwitch` and `Smoothie` has same price so we have to randomly sample these priced values.
- Another values like `Cake` and `Juice` has same Price and we will calculate these by same method.

# ****Menu Items****

| 🧾 **Item**     | 💲 **Price ($)** |
| --------------- | ---------------- |
| ☕ **Coffee**    | `2.00`           |
| 🍵 **Tea**      | `1.50`           |
| 🥪 **Sandwich** | `4.00`           |
| 🥗 **Salad**    | `5.00`           |
| 🍰 **Cake**     | `3.00`           |
| 🍪 **Cookie**   | `1.00`           |
| 🥤 **Smoothie** | `4.00`           |
| 🧃 **Juice**    | `3.00`           |

### 8.1 — Fill Unambiguous Items via Price Mapping

Four items have **unique prices** — meaning if the price is known, the item can be identified with certainty:

| Price | Item |
|---|---|
| $2.00 | Coffee |
| $1.50 | Tea |
| $5.00 | Salad |
| $1.00 | Cookie |

We build a reverse price-to-item dictionary and use `.map()` to look up the item name from the price. `.fillna()` ensures only missing `Item` rows are affected — existing values are untouched.

> **Note:** `Sandwich`/`Smoothie` ($4.00) and `Cake`/`Juice` ($3.00) are intentionally excluded here because their prices are shared — they cannot be identified from price alone and are handled separately below.

In [ ]:
def item_missing_cal(df):

    # Creating Price map
    price_map = {2.0 : 'Coffee', 1.50 : 'Tea', 5.0 : 'Salad', 1.0 : 'Cookie'}

    # Create replacement using map
    replace = df['Price Per Unit'].map(price_map)

    # Calculate missing using replacement
    df['Item'] = df['Item'].fillna(replace)

item_missing_cal(df)

### 8.2 — Fill Ambiguous Items at $3.00 via Random Sampling

`Cake` and `Juice` both cost $3.00. For rows where `Item` is missing and `Price Per Unit` is $3.00, we cannot determine which item it was.

**Solution:** Randomly assign `Cake` or `Juice` with equal 50/50 probability using `np.random.choice()`.

This is preferable to picking one value (e.g. always filling `Cake`) because that would over-represent one item and distort sales analysis downstream.

> **Limitation:** This introduces synthetic data for ambiguous rows. The assumption of a 50/50 split is reasonable without additional business data, but it should be documented as a known imputation decision.

In [ ]:
# For Items at price == 3.0

def missing_item_price_3(df):

    # Item are at price == 3.0
    maping = ['Cake', 'Juice']

    # Find rows that meet criteria
    masking = ((df['Item'].isnull()) & (df['Price Per Unit'] == 3))

    # Value Count of missing
    count = masking.sum()
    # Calculate missing
    if count > 0:
        df.loc[masking, 'Item'] = np.random.choice(maping, size = count, p = [0.5, 0.5])                   # Here p = [0.5, 0.5] means it will fill 50%-50% both values.


missing_item_price_3(df)

### 8.3 — Fill Ambiguous Items at $4.00 via Random Sampling

`Sandwich` and `Smoothie` both cost $4.00. The same random sampling approach is applied here.

Missing `Item` rows with `Price Per Unit == 4.0` are randomly assigned either `Sandwich` or `Smoothie` with equal probability.

> The `if count > 0` guard prevents `np.random.choice()` from being called with `size=0`, which would raise an error.

In [ ]:
# For Items at Price == 4.0

def missing_item_price_4(df):

    # Items at price 4
    items = ['Sandwich', 'Smoothie']

    # Find rows that meet criteria
    rows = ((df['Item'].isnull()) & (df['Price Per Unit'] == 4.0))

    # Count of these rows
    counts = rows.sum()

    # Calculate missing
    if counts > 0:
        df.loc[rows, 'Item'] = np.random.choice(items, size = counts, p = [0.5, 0.5])


missing_item_price_4(df)

## Step 9 — Impute Missing `Payment Method` and `Location` via Random Sampling

These are categorical columns with no mathematical relationship to other columns. Mean, median, and mode are not appropriate here.

**Why random sampling instead of mode?**
If the real distribution is 50% Cash, 30% Credit Card, 20% Digital Wallet — filling all missing values with `Cash` (the mode) would inflate Cash's share and distort any payment analysis. Random sampling draws replacements from the existing values, preserving the natural distribution proportionally.

**How it works:**
1. Identify the missing rows with `.isnull()`
2. Count how many are missing with `.sum()`
3. Randomly sample that many values from the non-null rows with `.sample()`
4. `.values` strips the index from the sampled Series to avoid index mismatch errors during assignment

In [ ]:
df['Payment Method'][df["Payment Method"].isnull()]=df['Payment Method'].dropna().sample(df['Payment Method'].isnull().sum()).values
df['Location'][df["Location"].isnull()]=df['Location'].dropna().sample(df['Location'].isnull().sum()).values

## Step 10 — Interpolate Missing `Transaction Date` Values

Dates cannot be interpolated directly — pandas cannot find "the midpoint between two dates" without first converting them to numbers.

**Approach — Convert → Interpolate → Convert back:**

1. **`.view('int64')`** — Every date is stored internally as the number of nanoseconds since January 1, 1970. This reinterprets the column as raw integers so arithmetic can be performed.

2. **`.where(df['Transaction Date'].notna(), np.nan)`** — `.view('int64')` cannot represent `NaN` (integers have no null concept), so missing positions become garbage large integers. This line restores `NaN` at the correct positions by referencing the original date column.

3. **`.interpolate(method='linear')`** — Fills each `NaN` integer by calculating the midpoint between its two neighboring values — the same way you would estimate a missing number in a sequence.

4. **`pd.to_datetime()`** — Converts the filled integers back to readable dates.

> **Assumption:** This method is reliable only when dates are in chronological order and gaps are small. Large gaps would produce interpolated dates that may not reflect real transaction timing.

In [ ]:
numeric_date = df['Transaction Date'].view('int64')

numeric_date = numeric_date.where(df['Transaction Date'].notna(), np.nan)

df['Transaction Date'] = pd.to_datetime(numeric_date.interpolate(method = 'linear'))

## Step 11 — Drop Rows Where Core Columns Are All Missing

At this point all recoverable values have been imputed. We now drop rows where `Item`, `Quantity`, and `Price Per Unit` are **all null simultaneously** — meaning there is no information left to recover or calculate from.

> `how='all'` means a row is only dropped if **every one** of the specified columns is null. A row missing just one of these columns would have been recovered earlier.

In [ ]:
df = df.dropna(subset = ['Item', 'Quantity', 'Price Per Unit'], how = 'all')

## Step 12 — Verify `Item` Column After Imputation

After all item imputation steps, we inspect the unique values in the `Item` column to confirm:
- All 8 valid menu items are present
- No `"ERROR"`, `"UNKNOWN"`, or `NaN` values remain

In [ ]:
df['Item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Juice',
       'Sandwich', 'Tea', nan], dtype=object)

## Step 13 — Final Pass: Fill Remaining `Price Per Unit` Gaps

After item imputation, every row now has a known `Item`. We use this to fill any remaining missing `Price Per Unit` values by mapping each item to its fixed price.

Once `Price Per Unit` is filled, we run `null_calculator()` one final time to recover any `Total Spent` or `Quantity` values that became calculable now that more prices are known.

In [ ]:
def priceperunit_missing(df):

    maping = {'Coffee' : 2.0, 'Cake' : 3.0, 'Cookie' : 1.0, 'Salad' : 5.0, 'Smoothie' : 4.0, 'Sandwich' : 4.0, 'Tea' : 1.50, 'Juice' : 3.0}

    replace = df['Item'].map(maping)

    df['Price Per Unit'] = df['Price Per Unit'].fillna(replace)

priceperunit_missing(df)
null_calculator(df)

## Step 14 — Null Check After All Imputation

We inspect the remaining null counts across all columns. Any nulls still present at this stage cannot be recovered — they will be reviewed individually before a final drop decision is made.

In [ ]:
df.isnull().sum()


,0
Transaction ID,0
Item,3
Quantity,20
Price Per Unit,3
Total Spent,23
Payment Method,0
Location,0
Transaction Date,0


## Step 15 — Inspect Remaining Null Rows Individually

Before dropping anything, we examine each column's remaining null rows to confirm that no further imputation is possible. This is an important step — blindly dropping rows without inspection risks losing recoverable data.

In [ ]:
df[df['Quantity'].isnull()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
236,TXN_8562645,Salad,NaN,5.0,NaN,Digital Wallet,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,NaN,3.0,NaN,Digital Wallet,Takeaway,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,Cash,Takeaway,2023-05-14
2796,TXN_9188692,Cake,NaN,3.0,NaN,Credit Card,In-store,2023-12-01
3203,TXN_4565754,Smoothie,NaN,4.0,NaN,Digital Wallet,Takeaway,2023-10-06
3224,TXN_6297232,Coffee,NaN,2.0,NaN,Digital Wallet,In-store,2023-04-07
3401,TXN_3251829,Tea,NaN,1.5,NaN,Digital Wallet,In-store,2023-07-25
4257,TXN_6470865,Coffee,NaN,2.0,NaN,Digital Wallet,Takeaway,2023-09-18
5841,TXN_5884081,Cookie,NaN,1.0,NaN,Digital Wallet,In-store,2023-07-05


In [ ]:
df[df['Item'].isna()]


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,Takeaway,2023-02-09
2289,TXN_7524977,NaN,4.0,NaN,NaN,Cash,In-store,2023-12-09
4152,TXN_9646000,NaN,2.0,NaN,NaN,Credit Card,In-store,2023-12-14


In [ ]:
df[df['Price Per Unit'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,Takeaway,2023-02-09
2289,TXN_7524977,NaN,4.0,NaN,NaN,Cash,In-store,2023-12-09
4152,TXN_9646000,NaN,2.0,NaN,NaN,Credit Card,In-store,2023-12-14


In [ ]:
df[df['Total Spent'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
236,TXN_8562645,Salad,NaN,5.0,NaN,Digital Wallet,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,NaN,3.0,NaN,Digital Wallet,Takeaway,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,Cash,Takeaway,2023-05-14
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,Takeaway,2023-02-09
2289,TXN_7524977,NaN,4.0,NaN,NaN,Cash,In-store,2023-12-09
2796,TXN_9188692,Cake,NaN,3.0,NaN,Credit Card,In-store,2023-12-01
3203,TXN_4565754,Smoothie,NaN,4.0,NaN,Digital Wallet,Takeaway,2023-10-06
3224,TXN_6297232,Coffee,NaN,2.0,NaN,Digital Wallet,In-store,2023-04-07
3401,TXN_3251829,Tea,NaN,1.5,NaN,Digital Wallet,In-store,2023-07-25


From above remaining missing data none of values can be calculated so we have to drop all these rows¶

## Step 16 — Drop Irrecoverable Rows

Having confirmed that the remaining null rows cannot be imputed, we drop them with `dropna()`. The final null check confirms the dataset is fully clean.

In [ ]:
df.dropna(inplace = True)
df.isnull().sum()

,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


## Step 17 — Data Cleaning Summary Report

A final summary comparing the raw and cleaned datasets, showing total rows removed and percentage data loss.

A low data loss percentage indicates the imputation strategies were effective — most missing values were recovered rather than dropped.

In [ ]:
# Calculate loss
rows_lost = original_data.shape[0] - df.shape[0]
percent_lost = (rows_lost / original_data.shape[0]) * 100

print(f"\033[1m{' DATA CLEANING SUMMARY ':^50}\033[0m")
print("=" * 50)

print(f"{'Metric':<25} | {'Raw Data':<10} | {'Cleaned':<10}")
print("-" * 50)
print(f"{'Number of Rows':<25} | {original_data.shape[0]:<10} | {df.shape[0]:<10}")
print(f"{'Number of Columns':<25} | {original_data.shape[1]:<10} | {df.shape[1]:<10}")

print("-" * 50)
print(f"\033[91m{'Total Rows Removed':<25} : {rows_lost}\033[0m")
print(f"\033[91m{'Data Loss Percentage':<25} : {percent_lost:.2f}%\033[0m")
print("=" * 50)

              DATA CLEANING SUMMARY               
Metric                    | Raw Data   | Cleaned   
--------------------------------------------------
Number of Rows            | 10000      | 9974      
Number of Columns         | 8          | 8         
--------------------------------------------------
Total Rows Removed        : 26
Data Loss Percentage      : 0.26%


In [ ]:
df.to_csv('cafe_sales_clean.csv', index=False)